# CDS catalogue client

Authenticate with the ECMWF Data Stores client, browse the catalogue, and inspect an ERA5 collection.

Run the cells in order. Credentials are read from local configuration; this notebook contains no saved credentials.

In [ ]:
import logging
import os
from datetime import datetime, timezone

from ecmwf.datastores import Client


## 1. Configure access


In [ ]:
# Leave blank to use credentials in ~/.ecmwfdatastoresrc.
# Never commit a notebook containing an API key.
api_key = ""

if api_key:
    os.environ["ECMWF_DATASTORES_KEY"] = api_key

# Show useful progress messages from the client.
logging.basicConfig(level=logging.INFO)


## 2. Authenticate


In [ ]:
# Create the client using saved ECMWF Data Stores credentials.
client = Client()
client.check_authentication()
print("Authentication successful.")


## 3. Browse the catalogue


In [ ]:
print("Querying catalogue...")

# List collection IDs, following all catalogue pages.
collections = client.get_collections(sortby="update")
collection_ids = []

while collections is not None:
    collection_ids.extend(collections.collection_ids)
    collections = collections.next

print("\nCatalogue collections:")
for number, collection_id in enumerate(collection_ids, start=1):
    print(f"{number}) {collection_id}")


## 4. Inspect an ERA5 collection


In [ ]:
collection_id = "reanalysis-era5-pressure-levels"
collection = client.get_collection(collection_id)
print(f"Inspecting {collection_id}:")

attributes = [
    "title",
    "published_at",
    "updated_at",
    "begin_datetime",
    "end_datetime",
    "bbox",
]

for attribute in attributes:
    value = getattr(collection, attribute, None)
    print(f"\t[+]{attribute}: {value}")

    # Calculate how many days ago the collection was last updated.
    if attribute == "updated_at" and value is not None:
        if isinstance(value, str):
            value = datetime.fromisoformat(value.replace("Z", "+00:00"))

        if value.tzinfo is None:
            value = value.replace(tzinfo=timezone.utc)

        age_days = (datetime.now(timezone.utc) - value).days
        print(f"\t\tDays since update: {age_days}")


## 5. Optionally download a small sample


In [ ]:
# Set to True to request and download a small ERA5 sample.
DOWNLOAD_EXAMPLE = False

if DOWNLOAD_EXAMPLE:
    fileout = "era5_sample.nc"
    request = {
        "product_type": ["reanalysis"],
        "variable": ["temperature"],
        "year": ["2022"],
        "month": ["01"],
        "day": ["01"],
        "time": ["00:00"],
        "pressure_level": ["1000"],
        "data_format": "netcdf",
        "download_format": "unarchived",
    }

    client.retrieve(collection_id, request, target=fileout)
    print(f"Downloaded: {fileout}")
